## 1. Catalog & Schema Architecture
We initialize our 3-level namespace boundary (`Catalog ➔ Schema ➔ Table`). This setup provisions isolated operational zones for our data as it progresses through the Medallion framework.

* **Catalog:** `fintech` (Central data governance hub)
* **Schemas:** 
    * `bronze`: Historical raw data dump (immutable strings).
    * `silver`: Cleaned, typed, and structured data tables.
    * `gold`: Business-level aggregates and reporting views.

In [0]:
USE CATALOG fintech;
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

## 2. Raw Ingestion Layer (Managed Volume)
To stage unstructured files like raw CSV data, we create a managed Unity Catalog **Volume**. This provides a direct, high-performance file-system directory inside our secure cloud boundary.

In [0]:
USE CATALOG fintech;
USE SCHEMA bronze;
CREATE VOLUME IF NOT EXISTS raw_data;

## 3. Data Ingestion Methods (Local Machine ➔ Databricks Volume)

To populate our newly created `raw_data` Volume with bank source files, we utilize one of two operational pipelines depending on whether the task is ad-hoc development or a scheduled production run:

### Method A: Manual UI Upload (Best for Ad-hoc Development)

1. Click the **Catalog** icon on the far left Databricks sidebar.
2. Navigate to: `fintech` ➔ `bronze` ➔ `Volumes` ➔ `raw_data`.
3. Click the **Upload to this volume** button in the top right.
4. Drag and drop local CSV files directly into the browser window.

### Method B: Programmatic CLI Upload (Best for Local Machine Automation)

For automated daily batch deliveries, we bypass the browser UI entirely. A local machine script or scheduled task pushes files directly into the cloud volume using the secure Databricks Command Line Interface (CLI).

Run this command inside your local machine terminal to stream files up to the volume:

```
databricks fs cp ./my_local_folder/transactions.csv dbfs:/Volumes/fintech/bronze/raw_data/
```

## 4. Landing Zone Audit
We execute a filesystem check directly on our Volume path to discover newly arrived CSV files and ensure our data transfer layer is communicating correctly.

In [0]:
LIST '/Volumes/fintech/bronze/raw_data'

## 5. Staged File Data Inspection
Before building permanent pipelines, we perform an ad-hoc data preview using the native `read_files` table function. 

> **Bronze Safety Rule:** We explicitly set `inferSchema => false` to force all incoming data fields to load as raw string data types. This prevents type parsing errors from crashing our pipelines during raw historical landing.

In [0]:
SELECT * FROM read_files(
    '/Volumes/fintech/bronze/raw_data/transactions.csv',
    format => 'csv',
    header => true,
    inferSchema => false
)
LIMIT 100;